# Recover from Kafka outages and duplicate delivery

**Goal.** Work through a bounded, reproducible example and inspect the evidence before connecting an external service.

**Prerequisites.** Base FraudTwin install. Optional extras and Docker commands are clearly marked.

**Produces.** Tables, fingerprints, manifests, and verification output.


**Source size.** The default cells generate approximately 1,000 logical payments; increase duration and population together for a 10,000-payment run.

**Offline path.** All marked offline cells run without Docker or network services. Service cells are optional and explicitly marked in notebook metadata.

**Cleanup.** Outputs are written under a temporary directory; remove any local run directory if you changed the output location.


**Set up a deterministic source run**


In [ ]:
# ruff: noqa
from pathlib import Path
from tempfile import TemporaryDirectory
import json
import polars as pl

from fraudtwin.config import load_config
from fraudtwin.generation import generate

root = next(
    (p for p in (Path.cwd(), *Path.cwd().parents) if (p / "configs" / "minimal.yaml").exists()),
    Path.cwd(),
)
base = load_config(root / "configs" / "minimal.yaml")
# Scale the population so the bounded example produces about 1,000 payments.
population = base.population.model_copy(
    update={
        "customers": 200,
        "accounts": 300,
        "cards": 240,
        "devices": 240,
        "pix_keys": 160,
        "merchants": 60,
    }
)
simulation = base.simulation.model_copy(update={"duration_days": 10})
fraud = base.fraud.model_copy(update={"enabled": True, "target_rate": 0.05})
config = base.model_copy(
    update={"population": population, "simulation": simulation, "fraud": fraud}
)
data = generate(config, write=False)
run_id = data.run_id
payments = pl.DataFrame([item.model_dump(mode="json") for item in data.behavior.payments])
print({"run_id": run_id, "payments": len(payments), "events": len(data.behavior.payment_events)})

**Inspect schema, grain, and counts**


In [ ]:
# ruff: noqa
from fraudtwin.kafka import publication_records
from fraudtwin.kafka_chaos import KafkaChaosConfig, KafkaChaosOutage, simulate_delivery

records = publication_records(data.behavior, run_id)[:80]
print({"sent": len(records), "topics": sorted({r.topic for r in records})})

**Run the core operation**


In [ ]:
# ruff: noqa
chaos_config = KafkaChaosConfig(
    seed=11,
    drop_probability=0.03,
    duplicate_probability=0.08,
    retry_probability=0.1,
    max_delay_seconds=60,
    reorder_window=10,
    partition_count=3,
    partition_skew_probability=0.2,
)
result = simulate_delivery(records, chaos_config)
print(result.manifest["counts"])

**Measure and interpret the result**


In [ ]:
# ruff: noqa
outage = KafkaChaosOutage(
    from_time=records[0].observable_time,
    to_time=records[-1].observable_time,
    behavior="BUFFER_AND_FLUSH",
)
outage_result = simulate_delivery(records, chaos_config.model_copy(update={"outages": (outage,)}))
print("outage counts:", outage_result.manifest["counts"])

**Exercise a parameter or failure mode**


In [ ]:
# ruff: noqa
deduped = {(e.subject, e.record_id): e for e in result.envelopes}
print({"delivered": len(result.envelopes), "deduplicated": len(deduped), "late": result.late_count})

**Write a compact artifact and fingerprint**


In [ ]:
# ruff: noqa
assert all(e.record_id for e in deduped.values())
print("stable event identity enables safe consumer deduplication")

**Verify invariants and clean up**


In [ ]:
# ruff: noqa
display(pl.DataFrame(result.audit[:12]))

**Optional service integration**


In [ ]:
# ruff: noqa
print(
    "Watermarks must account for delayed/out-of-order event time; physical packet loss is outside this logical harness."
)

**Review the expected outcome**


In [ ]:
# ruff: noqa
# A compact inspection is more useful than printing an entire run.
print(
    payments.select(
        [
            c
            for c in ("payment_id", "amount", "initiated_at", "payer_account_id")
            if c in payments.columns
        ]
    ).head(8)
)
print({"columns": payments.columns, "nulls": payments.null_count().to_dicts()[0]})

**Next recommended step**


In [ ]:
# ruff: noqa
summary = {
    "run_id": run_id,
    "payments": len(data.behavior.payments),
    "payment_events": len(data.behavior.payment_events),
    "fraud_records": len(data.behavior.fraud_records),
}
assert summary["payments"] == len(payments)
assert summary["payments"] > 0
print(json.dumps(summary, indent=2, default=str))

**Next recommended step**


In [ ]:
# ruff: noqa
print("Optional service cell: run the same envelopes through a local Kafka broker.")